# Does a frozen weighted measure bill diversifying flow?

Weighted-spread freezes `q(S)` at the first mint. This notebook asks whether that snapshot can drift far enough that an empty-book island — diversifying under live settlement — is billed like body risk, while a same-size open at live spot is cheap.

That is the failure that would make freeze-once a net negative: you charge the flow you want and miss the pile that actually matters.

`D = stdev_q(W)`. The charge is `r · ΔD`, signed, matching `at/dbu-732-weighted-spread`. Truth is the same `ΔD` under **live** `q`. Frozen `q` is the creation lognormal and never moves.

The book is piled at the creation ATM and then left alone. Three same-size tickets are quoted after decay and spot drift: a **far-wing island** (empty, 1.5 creation-σ, opposite any drift), a **shoulder island** (empty, 0.5 creation-σ, still in the frozen body), and a **live-ATM island** (empty, at today's forward). A pile-on at the existing book is the control — after drift that location is the leftover pile in a region live `q` has left.

Elapsed fractions and drift in creation-σ match the [K-grid freeze experiment](../docs/design/inventory_impact_freeze_policy_spec.md). Refresh is not a candidate here: weighted still rebates, so reweighting `q` moves `D` with no trade.

**Decision rule.** Freeze-once is a problem if, inside the reachable region (elapsed ≤ 90%, |drift| ≤ 1 creation-σ), the frozen charge on the wing island exceeds the frozen charge on the live-ATM island **and** live `q` ranks them the other way. That is a rank inversion: diversifying costs more than concentrating where settlement will happen. Overcharge of the wing without inversion is inefficiency, not a net negative. Undercharge of the live ATM without inversion is the K-grid staleness (uncompensated risk), reported but not this question.

This is a parametric lognormal (and one fat-tail check). It does not use BTC, Move rounding, or trader relocation beyond the three quoted tickets.

In [ ]:
from __future__ import annotations

import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CELLS = 2_000
SPAN_SIGMA = 4.0
TICKET_CELLS = 20
TICKET_QTY = 17_000.0
PILE_CELLS = 80
PILE_QTY = 40_000.0
RATE = 0.181599
WING_OFFSET = 1.5
SHOULDER_OFFSET = 0.5
ELAPSED = (0.0, 0.25, 0.50, 0.75, 0.90, 0.99)
DRIFTS = (-2.0, -1.0, 0.0, 1.0, 2.0)
REACH_ELAPSED = 0.90
REACH_DRIFT = 1.0
T_DF = 4.0

Z = (np.arange(CELLS) + 0.5) / CELLS * (2.0 * SPAN_SIGMA) - SPAN_SIGMA


def cell_at(z: float) -> int:
    return int(np.clip(np.argmin(np.abs(Z - z)), 0, CELLS - 1))


def span(center_z: float, width: int) -> tuple[int, int]:
    mid = cell_at(center_z)
    lo = int(np.clip(mid - width // 2, 0, CELLS - width))
    return lo, lo + width


def normal_pdf(z: np.ndarray) -> np.ndarray:
    return np.exp(-0.5 * z * z) / math.sqrt(2.0 * math.pi)


def student_t_pdf(z: np.ndarray, df: float) -> np.ndarray:
    const = math.gamma((df + 1.0) / 2.0) / (math.sqrt(df * math.pi) * math.gamma(df / 2.0))
    return const * np.power(1.0 + (z * z) / df, -(df + 1.0) / 2.0)


def measure(elapsed: float, drift: float, law: str) -> np.ndarray:
    remaining = max(1.0 - elapsed, 1e-12)
    width = float(np.sqrt(remaining))
    z_live = (Z - drift) / width
    if law == "lognormal":
        dens = normal_pdf(z_live)
    elif law == "student-t":
        scale = float(math.sqrt(T_DF / (T_DF - 2.0)))
        dens = student_t_pdf(z_live / scale, T_DF) / scale
    else:
        raise ValueError(law)
    q = np.maximum(dens / width, 0.0)
    return q / q.sum()


def deviation(payout: np.ndarray, q: np.ndarray) -> float:
    mean = float(np.dot(q, payout))
    second = float(np.dot(q, payout * payout))
    return float(np.sqrt(max(second - mean * mean, 0.0)))


def charge(payout: np.ndarray, lo: int, hi: int, qty: float, q: np.ndarray) -> float:
    after = payout.copy()
    after[lo:hi] += qty
    return RATE * (deviation(after, q) - deviation(payout, q))


def book() -> np.ndarray:
    payout = np.zeros(CELLS)
    lo, hi = span(0.0, PILE_CELLS)
    payout[lo:hi] += PILE_QTY
    return payout


def tickets(drift: float) -> dict[str, tuple[int, int]]:
    side = -float(np.sign(drift)) if abs(drift) > 1e-12 else 1.0
    return {
        "pile_on": span(0.0, TICKET_CELLS),
        "stale_wing": span(side * WING_OFFSET, TICKET_CELLS),
        "shoulder": span(side * SHOULDER_OFFSET, TICKET_CELLS),
        "live_atm": span(drift, TICKET_CELLS),
    }


Q0 = measure(0.0, 0.0, "lognormal")
PAYOUT = book()
print(
    f"creation D ${deviation(PAYOUT, Q0):,.0f}  "
    f"pile cells {span(0.0, PILE_CELLS)}  "
    f"ticket {TICKET_CELLS} cells × ${TICKET_QTY:,.0f}  r={RATE:.4%}"
)

In [ ]:
def sweep(law: str) -> pd.DataFrame:
    q_frozen = measure(0.0, 0.0, law)
    rows = []
    for elapsed in ELAPSED:
        for drift in DRIFTS:
            q_live = measure(elapsed, drift, law)
            spans = tickets(drift)
            row = {
                "law": law,
                "elapsed": elapsed,
                "drift": drift,
                "reachable": elapsed <= REACH_ELAPSED and abs(drift) <= REACH_DRIFT,
                "same_ticket": spans["live_atm"] == spans["pile_on"],
            }
            for name, (lo, hi) in spans.items():
                frozen = charge(PAYOUT, lo, hi, TICKET_QTY, q_frozen)
                live = charge(PAYOUT, lo, hi, TICKET_QTY, q_live)
                row[f"{name}_frozen"] = frozen
                row[f"{name}_live"] = live
            row["wing_inverted"] = (
                (not row["same_ticket"])
                and row["stale_wing_frozen"] > row["live_atm_frozen"]
                and row["live_atm_live"] > row["stale_wing_live"]
            )
            row["shoulder_inverted"] = (
                (not row["same_ticket"])
                and row["shoulder_frozen"] > row["live_atm_frozen"]
                and row["live_atm_live"] > row["shoulder_live"]
            )
            row["rank_inverted"] = bool(row["wing_inverted"] or row["shoulder_inverted"])
            rows.append(row)
    return pd.DataFrame(rows)


def money(df: pd.DataFrame, col: str) -> pd.DataFrame:
    table = df.pivot(index="elapsed", columns="drift", values=col)
    return table.map(lambda x: f"${x:,.1f}")


LOGN = sweep("lognormal")
FAT = sweep("student-t")

print("Frozen $ on the far-wing island (1.5σ, empty)")
display(money(LOGN, "stale_wing_frozen"))
print("Live-q $ on that far-wing island")
display(money(LOGN, "stale_wing_live"))
print("Frozen $ on the shoulder island (0.5σ, empty, still in the frozen body)")
display(money(LOGN, "shoulder_frozen"))
print("Live-q $ on that shoulder island")
display(money(LOGN, "shoulder_live"))
print("Frozen $ on the live-ATM island")
display(money(LOGN, "live_atm_frozen"))
print("Live-q $ on the live-ATM island (truth)")
display(money(LOGN, "live_atm_live"))

In [ ]:
print("Pile-on at the existing book, frozen $ (control; does not move with drift)")
display(money(LOGN, "pile_on_frozen"))
print("Pile-on at the existing book, live-q $")
display(money(LOGN, "pile_on_live"))

reach = LOGN[LOGN["reachable"]].copy()
moved = reach[~reach["same_ticket"]]
inversions = int(moved["rank_inverted"].sum())
wing_over = (reach["stale_wing_frozen"] - reach["stale_wing_live"]).mean()
atm_under = (reach["live_atm_live"] - reach["live_atm_frozen"]).mean()

print()
print(f"reachable cells (elapsed ≤ {REACH_ELAPSED:.0%}, |drift| ≤ {REACH_DRIFT:.0f}σ): {len(reach)}")
print(f"far-wing inversions among drifted cells: {int(moved['wing_inverted'].sum())} / {len(moved)}")
print(f"shoulder inversions among drifted cells: {int(moved['shoulder_inverted'].sum())} / {len(moved)}")
print(f"mean frozen − live on the far-wing island: ${(reach['stale_wing_frozen'] - reach['stale_wing_live']).mean():,.1f}")
print(f"mean frozen − live on the shoulder island: ${(reach['shoulder_frozen'] - reach['shoulder_live']).mean():,.1f}")
print(f"mean live − frozen on the live-ATM island: ${atm_under:,.1f}  (positive = miss real risk)")
print(f"mean frozen − live on the leftover pile-on: ${(reach['pile_on_frozen'] - reach['pile_on_live']).mean():,.1f}  (positive = billing a now-dead pile)")
print()
print(moved[
    ["elapsed", "drift", "stale_wing_frozen", "shoulder_frozen",
     "live_atm_frozen", "live_atm_live", "pile_on_frozen", "pile_on_live",
     "wing_inverted", "shoulder_inverted"]
].to_string(index=False, formatters={
    "stale_wing_frozen": "${:,.1f}".format,
    "shoulder_frozen": "${:,.1f}".format,
    "live_atm_frozen": "${:,.1f}".format,
    "live_atm_live": "${:,.1f}".format,
    "pile_on_frozen": "${:,.1f}".format,
    "pile_on_live": "${:,.1f}".format,
}))

print()
print("Student-t(4), matched variance — rank inversions in the reachable drifted region")
fat_moved = FAT[FAT["reachable"] & ~FAT["same_ticket"]]
print(f"{int(fat_moved['rank_inverted'].sum())} / {len(fat_moved)}")

In [ ]:
def heatmap(ax, df, col, title, cmap, vmin, vmax):
    grid = df.pivot(index="elapsed", columns="drift", values=col)
    image = ax.imshow(
        grid.to_numpy(),
        origin="lower",
        aspect="auto",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_xticks(range(len(grid.columns)), [f"{c:g}" for c in grid.columns])
    ax.set_yticks(range(len(grid.index)), [f"{r:.0%}" for r in grid.index])
    ax.set_xlabel("spot drift, creation-σ")
    ax.set_ylabel("elapsed life")
    ax.set_title(title)
    for y, elapsed in enumerate(grid.index):
        for x, drift in enumerate(grid.columns):
            value = grid.loc[elapsed, drift]
            ax.text(x, y, f"{value:.0f}", ha="center", va="center", fontsize=8)
    return image


figure, axes = plt.subplots(1, 3, figsize=(14.2, 4.2))
vmax = max(
    LOGN["stale_wing_frozen"].abs().max(),
    LOGN["live_atm_frozen"].abs().max(),
    LOGN["live_atm_live"].abs().max(),
)
heatmap(axes[0], LOGN, "pile_on_frozen", "Frozen $  ·  leftover pile-on", "Reds", 0.0, vmax)
heatmap(axes[1], LOGN, "live_atm_frozen", "Frozen $  ·  live-ATM island", "Reds", 0.0, vmax)
image = heatmap(axes[2], LOGN, "live_atm_live", "Live-q $  ·  live-ATM island (truth)", "Reds", 0.0, vmax)
figure.colorbar(image, ax=axes, fraction=0.02, pad=0.02, label="inventory charge ($)")
figure.suptitle(
    "Same $17k ticket. Left is adding to the leftover pile (frozen q still thinks it is the body). "
    "Middle is an empty open at live spot under freeze-once. Right is what live q would bill there.",
    y=1.03,
    fontsize=11,
)
plt.show()

print(
    "A 1h market created three periods out is at ~67% elapsed when its live hour starts. "
    "A 1m market reaches 90% elapsed in 54 seconds. "
    "Decay alone (drift 0) is the clean test of leftover wing mass; drift is the misplaced-centre test."
)

## Result

Empty-island diversifying is not billed a ton. The far wing is ~$8 frozen at every horizon; the 0.5σ shoulder is ~$21. After decay, live `q` bills those islands ~$0, so freeze-once overcharges them by those same small dollars. That is the feared sign, not the feared size — both are a few percent of the $300 leftover pile-on.

Frozen charges do not grow with time. `q` is stuck, so leftover wing mass cannot start charging more later. The overcharge is the creation-time island fee, held constant while truth falls to zero.

The large wrong-area charge is adding to the leftover pile after spot has moved. Freeze-once still bills that pile-on $300 when live `q` would bill $0 (2σ, 90% elapsed) or $39 (1σ, 90%). That is charging a now-dead region at full body rate.

The large miss is the empty open at live spot. Freeze-once bills $15 at 1σ and $3 at 2σ; live `q` bills $47–$570 in the reachable region and ~$900+ at 2σ late in life. Shoulder-vs-live-ATM rank inversion appears at every drifted point for the 0.5σ island ($21 > $14) and only at 2σ for the far wing ($8 > $3). Both inversions are cheap-vs-cheaper. Informed flow picks the live ATM because it is the cheaper frozen ticket, which is the uncompensated-risk side, not the overbilled-diversifier side.

Student-t(4) does not add inversions beyond the lognormal drifted set.

Freeze-once is not a net negative from hammering diversifying islands. It is the same staleness the K-grid already measured: you keep billing yesterday's body and you miss today's. Do not refresh while rebates exist. If a far 1h needs a moving measure, drop the rebate first.

## Non-claims

The settlement law is parametric. Nothing here is a BTC calibration.

The book is one pile. Mixed flow, several islands, or a book that was itself built after drift are not scored.

Tickets are a fixed dollar and a fixed log-width, not a fixed live probability mass.

The charge uses the branch rate from the equal-prob skew fit so dollars match the placement figure. Rank inversion does not depend on that rate.

This does not recommend a refresh cadence. Weighted rebates; a moving measure is an extraction surface until the rebate is removed.